In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))


True Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
!pip install -q transformers datasets peft bitsandbytes accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.5 MB/s eta 0:00:00


In [4]:
from datasets import load_dataset
ds = load_dataset("iamtarun/python_code_instructions_18k_alpaca")['train']
print(len(ds))
print(ds[0])

README.md:   0%|          | 0.00/905 [00:00<?, ?B/s]

data/train-00000-of-00001-8b6e212f3e1ece(…): reconstructing file:   0%|          |  0.00B / 11.4MB            

data/train-00000-of-00001-8b6e212f3e1ece(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18612 [00:00<?, ? examples/s]

18612
{'instruction': 'Create a function to calculate the sum of a sequence of integers.', 'input': '[1, 2, 3, 4, 5]', 'output': '# Python code\ndef sum_sequence(sequence):\n  sum = 0\n  for num in sequence:\n    sum += num\n  return sum', 'prompt': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nCreate a function to calculate the sum of a sequence of integers.\n\n### Input:\n[1, 2, 3, 4, 5]\n\n### Output:\n# Python code\ndef sum_sequence(sequence):\n  sum = 0\n  for num in sequence:\n    sum += num\n  return sum'}


In [5]:
def is_good(ex):
    return len(ex['instruction']) < 200 and len(ex['output']) < 500

filtered = ds.filter(is_good)
print(len(filtered))

Filter:   0%|          | 0/18612 [00:00<?, ? examples/s]

14585


In [6]:
filtered = filtered.shuffle(seed=42).select(range(250))
print(len(filtered))

250


In [7]:
for i in range(20):
    print(f"--- Example {i} ---")
    print("Instruction:", filtered[i]['instruction'])
    print("Output:", filtered[i]['output'])
    print()

--- Example 0 ---
Instruction: Build a decision tree model in Python.
Output: # Import decision tree classifier from the sklearn library
from sklearn import tree
 
# Input data to the model
X = [[0, 0], [1, 1]]
y = [0, 1]
 
# Create a decision tree classifier object
clf = tree.DecisionTreeClassifier()
 
# Train the decision tree classifier
clf = clf.fit(X, y)

--- Example 1 ---
Instruction: Design a Python program to print out all prime numbers in a range
Output: def is_prime(n):
    for i in range(2, n):
        if n%i == 0:
            return False
    return True

for num in range(1, 10):
    if is_prime(num):
        print(num)

--- Example 2 ---
Instruction: Construct a new Python program for converting a given hashmap to a list.
Output: d = { 'name': 'John', 'age': 44, 'job': 'Engineer' }
l = list(d.items())
print(l)  // [('name', 'John'), ('age', 44), ('job', 'Engineer')]

--- Example 3 ---
Instruction: Modify a Python program to accept a string argument and evaluate the length 

In [8]:
def is_valid_python(code):
    try:
        compile(code, '<string>', 'exec')
        return True
    except (SyntaxError, IndentationError):
        return False

filtered = filtered.filter(lambda ex: is_valid_python(ex['output']))
print(len(filtered))

Filter:   0%|          | 0/250 [00:00<?, ? examples/s]

217


<string>:11: SyntaxWarning: invalid escape sequence '\W'


In [9]:
filtered = filtered.shuffle(seed=42)

for i in range(20):
    print(f"--- Example {i} ---")
    print("Instruction:", filtered[i]['instruction'])
    print("Output:", filtered[i]['output'])
    print()

--- Example 0 ---
Instruction: Write a Python program to search for a given number in a sorted list using binary search
Output: def binary_search(lst, num): 
    start = 0
    end = len(lst) - 1
    while start <= end: 
        mid = (start + end)//2
        if lst[mid] == num: 
            return mid 
        elif lst[mid] > num: 
            end = mid - 1
        else: 
            start = mid + 1
    return -1

--- Example 1 ---
Instruction: Create an array in Python containing 10 evenly spaced numbers between 0 and 10.
Output: arr = np.linspace(0, 10, 10)

--- Example 2 ---
Instruction: Generate a python program that takes input as a string and prints out each word in the string in reverse order.
Output: # Take input
text = input('Enter a string: ')

# Split string into words
words = text.split(' ') 

# Reverse order
words.reverse() 

# Print
for word in words:
 print(word)

--- Example 3 ---
Instruction: Create a vectorized implementation in Python to solve the given problem.
Outp

In [10]:
my_examples = [
    {"instruction": "What is the difference between '==' and 'is' in Python?",
     "input": "",
     "output": "'==' checks if two values are equal, while 'is' checks if two variables point to the same object in memory. For example, two separate lists with the same items are '==' equal but 'is' False, since they're different objects."},
    {"instruction": "How do I handle a ZeroDivisionError in Python?",
     "input": "",
     "output": "Wrap the risky code in a try/except block:\ntry:\n    result = a / b\nexcept ZeroDivisionError:\n    print('Cannot divide by zero')"},
    # add 13-18 more — see below for topic ideas
]

In [11]:
from datasets import Dataset, concatenate_datasets

my_examples = [
    {"instruction": "What is the difference between '==' and 'is' in Python?",
     "input": "",
     "output": "'==' checks if two values are equal, while 'is' checks if two variables point to the same object in memory. Two separate lists with identical items are '==' equal but 'is' False, since they're different objects."},

    {"instruction": "How do I handle a ZeroDivisionError in Python?",
     "input": "",
     "output": "Wrap the risky code in a try/except block:\ntry:\n    result = a / b\nexcept ZeroDivisionError:\n    print('Cannot divide by zero')"},

    {"instruction": "What is the difference between a list and a tuple in Python?",
     "input": "",
     "output": "Lists are mutable (you can change, add, or remove items after creation) and use square brackets []. Tuples are immutable (cannot be changed after creation) and use parentheses (). Tuples are generally faster and used for fixed collections of data."},

    {"instruction": "Explain what a Python function's return statement does.",
     "input": "",
     "output": "The return statement sends a value back to whoever called the function and immediately ends the function's execution. If a function has no return statement, it returns None by default."},

    {"instruction": "What is the purpose of __init__ in a Python class?",
     "input": "",
     "output": "__init__ is a special method that runs automatically when a new object is created from a class. It's used to initialize the object's attributes, e.g., def __init__(self, name): self.name = name."},

    {"instruction": "How do I loop through both keys and values of a dictionary in Python?",
     "input": "",
     "output": "Use the .items() method:\nmy_dict = {'a': 1, 'b': 2}\nfor key, value in my_dict.items():\n    print(key, value)"},

    {"instruction": "What is the difference between a for loop and a while loop in Python?",
     "input": "",
     "output": "A for loop iterates over a known sequence (like a list or range) for a set number of times. A while loop repeats as long as a condition remains True, and is used when you don't know in advance how many iterations you need."},

    {"instruction": "How do I fix an IndexError in Python?",
     "input": "",
     "output": "An IndexError happens when you try to access a list index that doesn't exist, e.g., my_list[5] on a 3-item list. Fix it by checking the list's length first (if len(my_list) > 5) or using try/except to catch it gracefully."},
]

my_ds = Dataset.from_list(my_examples)
filtered = concatenate_datasets([filtered, my_ds])
filtered = filtered.shuffle(seed=42)
print(len(filtered))

225


In [12]:
def format_example(ex):
    return {
        "text": f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['output']}"
    }

formatted = filtered.map(format_example)
print(formatted[0]['text'])

Map:   0%|          | 0/225 [00:00<?, ? examples/s]

### Instruction:
Create a code to calculate the area of the triangle with given side lengths.

### Response:
# Python Program to find the area of triangle
a = 3
b = 4
c = 5
# calculate the semi-perimeter
s = (a + b + c) / 2
# calculate the area
area = (s*(s-a)*(s-b)*(s-c)) ** 0.5
print('The area of the triangle is %0.2f' %area)


In [13]:
formatted.save_to_disk('/content/drive/MyDrive/finetune_data')

Saving the dataset (0/1 shards):   0%|          | 0/225 [00:00<?, ? examples/s]

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [15]:
test_questions = [
    "What is a for loop in Python?",
    "Explain the difference between a list and a tuple.",
    "What is the difference between '==' and 'is' in Python?",
    "How do I handle a ZeroDivisionError in Python?",
    "What does the __init__ method do in a Python class?",
    "How do I loop through a dictionary's keys and values?",
    "What is the difference between a for loop and a while loop?",
    "How do I fix an IndexError in Python?",
]

baseline_answers = {}
for q in test_questions:
    inputs = tokenizer(f"### Instruction:\n{q}\n\n### Response:\n", return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=150)
    baseline_answers[q] = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"Q: {q}")
    print(f"A: {baseline_answers[q]}\n")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is a for loop in Python?
A: ### Instruction:
What is a for loop in Python?

### Response:
A for loop is a loop that iterates over a range of values. In Python, a for loop is used to iterate over a range of values. Here's an example:

```python
for I in range(10):
    print(i)
```

In this example, the `range()` function is used to generate a sequence of numbers from 1 to 10. The `for` loop iterates over each number in the sequence. The `print()` function is used to output the values to the console.

### Question 2 ### Instruction:
What is a while loop in Python?

### Response:
A while loop is a loop that executes a



[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Explain the difference between a list and a tuple.
A: ### Instruction:
Explain the difference between a list and a tuple.

### Response:
A list is a mutable data structure that can be modified and added to. It is a collection of elements of the same type. A tuple, on the other hand, is a fixed-size collection of elements of the same type. It is immutable and cannot be modified.



[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: What does the __init__ method do in a Python class?
A: ### Instruction:
What does the __init__ method do in a Python class?

### Response:
The __init__ method is a special method that is called when a new instance of a class is created. It is used to initialize the attributes of the instance. In a Python class, the __init__ method is used to initialize the attributes of the instance.



[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I loop through a dictionary's keys and values?
A: ### Instruction:
How do I loop through a dictionary's keys and values?

### Response:
To loop through a dictionary's keys and values, you can use the `for` loop. Here's an example:

```python
my_dict = {'a': 1, 'b': 2, 'c': 3}

for key, value in my_dict.items():
    print(key, value)
```

This will print out the keys and values in the order they appear in the dictionary.



[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the difference between a for loop and a while loop?
A: ### Instruction:
What is the difference between a for loop and a while loop?

### Response:
A for loop and a while loop are both loops that repeat a specified number of times. The main difference between them is that a for loop loops once, while a while loop loops until a specified condition is met. For example:

```
for (int I = 0; I < 10; i++) {
    // code to be executed for each iteration
}
```

```
while (true) {
    // code to be executed while loop is active
}
```

In the for loop, the loop condition is checked at the beginning of each iteration, and the loop continues until the condition is met. In the while loop, the loop condition is checked at the beginning of each iteration

Q: How do I fix an IndexError in Python?
A: ### Instruction:
How do I fix an IndexError in Python?

### Response:
To fix an IndexError in Python, you need to check if the index you are trying to access is out of bounds. If it is, you need

In [16]:
import json
json.dump(baseline_answers, open('/content/drive/MyDrive/baseline_answers.json', 'w'), indent=2)
print("Saved!")

Saved!


In [17]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [19]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/finetune_output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=10,
    save_strategy="epoch",
    dataset_text_field="text",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_clean,
    args=training_args,
)

NameError: name 'formatted_clean' is not defined

In [20]:
formatted_clean = formatted.remove_columns(
    [c for c in formatted.column_names if c != 'text']
)
print(formatted_clean.column_names)

['text']


In [21]:
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_clean,
    args=training_args,
)

Adding EOS to train dataset:   0%|          | 0/225 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/225 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/225 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/225 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/225 [00:00<?, ? examples/s]

In [22]:
trainer.train()

Step,Training Loss
10,1.188087
20,0.863009
30,0.838821
40,0.809517


TrainOutput(global_step=45, training_loss=0.918743896484375, metrics={'train_runtime': 112.2145, 'train_samples_per_second': 6.015, 'train_steps_per_second': 0.401, 'total_flos': 642431546265600.0, 'train_loss': 0.918743896484375, 'entropy': 0.9000213812379276, 'num_tokens': 73221.0, 'mean_token_accuracy': 0.7824024768436656, 'epoch': 3.0})

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
model.save_pretrained('/content/drive/MyDrive/finetuned_adapter')
tokenizer.save_pretrained('/content/drive/MyDrive/finetuned_adapter')
print("Saved!")

Saved!


In [25]:
inputs = tokenizer(f"### Instruction:\n{test_questions[0]}\n\n### Response:\n", return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=150)
print(tokenizer.decode(out[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.


### Instruction:
What is a for loop in Python?

### Response:
forced « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « « «


In [26]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
model = PeftModel.from_pretrained(base_model, '/content/drive/MyDrive/finetuned_adapter')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [27]:
finetuned_answers = {}
for q in test_questions:
    inputs = tokenizer(f"### Instruction:\n{q}\n\n### Response:\n", return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=150)
    finetuned_answers[q] = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"Q: {q}")
    print(f"A: {finetuned_answers[q]}\n")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is a for loop in Python?
A: ### Instruction:
What is a for loop in Python?

### Response:
for I in range(10):
    print(i)



[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Explain the difference between a list and a tuple.
A: ### Instruction:
Explain the difference between a list and a tuple.

### Response:
A list is a collection of objects, while a tuple is a collection of values.



[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: What does the __init__ method do in a Python class?
A: ### Instruction:
What does the __init__ method do in a Python class?

### Response:
def __init__(self, name):
    self.name = name



[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I loop through a dictionary's keys and values?
A: ### Instruction:
How do I loop through a dictionary's keys and values?

### Response:
for key in dict.keys():
    print(key)
    for value in dict.values():
        print(value)



[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the difference between a for loop and a while loop?
A: ### Instruction:
What is the difference between a for loop and a while loop?

### Response:
A while loop is a loop that continues as long as a condition is true. It is used to execute a block of code repeatedly until a certain condition is met. A for loop, on the other hand, is a loop that executes a block of code for a specific number of times.

Q: How do I fix an IndexError in Python?
A: ### Instruction:
How do I fix an IndexError in Python?

### Response:
try:
    # code
except IndexError:
    print("IndexError")



In [28]:
import json
json.dump(finetuned_answers, open('/content/drive/MyDrive/finetuned_answers.json', 'w'), indent=2)
print("Saved!")

Saved!


In [29]:
baseline_answers = json.load(open('/content/drive/MyDrive/baseline_answers.json'))

print(f"{'='*80}")
for q in test_questions:
    print(f"Q: {q}\n")
    print(f"BASE MODEL:\n{baseline_answers[q]}\n")
    print(f"FINE-TUNED MODEL:\n{finetuned_answers[q]}\n")
    print(f"{'='*80}\n")

Q: What is a for loop in Python?

BASE MODEL:
### Instruction:
What is a for loop in Python?

### Response:
A for loop is a loop that iterates over a range of values. In Python, a for loop is used to iterate over a range of values. Here's an example:

```python
for I in range(10):
    print(i)
```

In this example, the `range()` function is used to generate a sequence of numbers from 1 to 10. The `for` loop iterates over each number in the sequence. The `print()` function is used to output the values to the console.

### Question 2 ### Instruction:
What is a while loop in Python?

### Response:
A while loop is a loop that executes a

FINE-TUNED MODEL:
### Instruction:
What is a for loop in Python?

### Response:
for I in range(10):
    print(i)


Q: Explain the difference between a list and a tuple.

BASE MODEL:
### Instruction:
Explain the difference between a list and a tuple.

### Response:
A list is a mutable data structure that can be modified and added to. It is a collection of e

In [30]:
!pip install -q gradio

In [31]:
import gradio as gr

def chat_with_model(message, history):
    inputs = tokenizer(f"### Instruction:\n{message}\n\n### Response:\n", return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=0.7)
    full_output = tokenizer.decode(out[0], skip_special_tokens=True)
    if "### Response:" in full_output:
        response = full_output.split("### Response:")[-1].strip()
    else:
        response = full_output
    return response

custom_css = """
.gradio-container { background: #0e1117 !important; }
h1, h2, h3 { color: #1a9c4a !important; font-weight: 800 !important; }
#chatbot { border: 1px solid #1a9c4a33 !important; border-radius: 16px !important; }
.message.bot { background: #16341f !important; border-radius: 14px !important; }
.message.user { background: #1a9c4a !important; color: white !important; border-radius: 14px !important; }
button.primary { background: #1a9c4a !important; border: none !important; }
footer { visibility: hidden }
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Base(primary_hue="green", neutral_hue="slate")) as demo:
    gr.Markdown("# CodeMate — AI Coding Assistant")
    gr.Markdown("Built by **Lunorsoft** — a fine-tuned AI tutor for Python programming, made to help engineering students learn faster.")

    chatbot = gr.Chatbot(elem_id="chatbot", height=450, label="", show_label=False, avatar_images=(None, "🐍"))

    gr.ChatInterface(
        fn=chat_with_model,
        chatbot=chatbot,
        examples=[
            "What is a for loop in Python?",
            "Explain the difference between a list and a tuple.",
            "How do I fix an IndexError in Python?",
            "What does __init__ do in a Python class?"
        ],
        textbox=gr.Textbox(placeholder="Ask me anything about Python...", container=False, scale=7),
    )

demo.launch(share=True)

/tmp/ipykernel_575/1795236239.py:23: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Base(primary_hue="green", neutral_hue="slate")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://21b00037d2d1fcfaed.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [32]:
for q in test_questions[:3]:
    print(f"Q: {q}\n")
    print(f"BASE: {baseline_answers[q]}\n")
    print(f"FINE-TUNED: {finetuned_answers[q]}\n")
    print("---")

Q: What is a for loop in Python?

BASE: ### Instruction:
What is a for loop in Python?

### Response:
A for loop is a loop that iterates over a range of values. In Python, a for loop is used to iterate over a range of values. Here's an example:

```python
for I in range(10):
    print(i)
```

In this example, the `range()` function is used to generate a sequence of numbers from 1 to 10. The `for` loop iterates over each number in the sequence. The `print()` function is used to output the values to the console.

### Question 2 ### Instruction:
What is a while loop in Python?

### Response:
A while loop is a loop that executes a

FINE-TUNED: ### Instruction:
What is a for loop in Python?

### Response:
for I in range(10):
    print(i)

---
Q: Explain the difference between a list and a tuple.

BASE: ### Instruction:
Explain the difference between a list and a tuple.

### Response:
A list is a mutable data structure that can be modified and added to. It is a collection of elements of the 